# 12 — Matrices and Linear Transformations

**Master syllabus coverage:** V2 2.3–2.4

## Why this matters

Every embedding lookup, projection, attention transform, and MLP layer depends on linear algebra. Shape derivation and rank reasoning reveal what these layers can preserve or discard.

## Learning objectives

- Implement matrix multiplication and interpret it as composition.
- Derive the shapes and orientation of an affine neural-network layer.
- Distinguish matrix rank from tensor rank/ndim.
- Connect basis choice and conditioning to numerical sensitivity.

Work through the notebook in order. Predict shapes and results before running each code cell. An assertion failure is part of the lesson: read it before changing code.


## 1. A matrix is a linear map

For $A\in\mathbb{R}^{m\times n}$ and $x\in\mathbb{R}^n$, $Ax\in\mathbb{R}^m$.
Columns show where input basis vectors go. Rows define output coordinates as dot
products with the input.

Matrix multiplication composes maps: if $B:\mathbb{R}^k\to\mathbb{R}^n$ and
$A:\mathbb{R}^n\to\mathbb{R}^m$, then $AB:\mathbb{R}^k\to\mathbb{R}^m$.


In [ ]:
import numpy as np
import torch

def matmul(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    assert a.ndim == 2 and b.ndim == 2 and a.shape[1] == b.shape[0]
    out = np.zeros((a.shape[0], b.shape[1]), dtype=np.result_type(a, b))
    for i in range(a.shape[0]):
        for j in range(b.shape[1]):
            for k in range(a.shape[1]):
                out[i, j] += a[i, k] * b[k, j]
    return out

A = np.array([[2.0, 0.0], [0.0, 0.5]])
B = np.array([[1.0, -1.0], [1.0, 1.0]])
np.testing.assert_allclose(matmul(A, B), A @ B)
print("A @ B =\n", A @ B)
print("B @ A =\n", B @ A)
assert not np.allclose(A @ B, B @ A)


## 2. Affine layers

Neural-network “linear” layers usually include a bias and are therefore affine:

$$Y=XW^\top+b$$

Shapes: $X=[B,C_{in}]$, $W=[C_{out},C_{in}]$, $b=[C_{out}]$,
$Y=[B,C_{out}]$. PyTorch stores `Linear.weight` in the displayed $W$ orientation.


In [ ]:
torch.manual_seed(42)
X = torch.randn(3, 4)       # [B,Cin]
W = torch.randn(5, 4)       # [Cout,Cin]
b = torch.randn(5)          # [Cout]
manual = X @ W.T + b        # [B,Cout]

layer = torch.nn.Linear(4, 5)
with torch.no_grad():
    layer.weight.copy_(W)
    layer.bias.copy_(b)
torch.testing.assert_close(manual, layer(X))
print("output shape:", manual.shape)


## 3. Rank and information capacity

Matrix rank is the number of independent row/column directions. A map with rank less
than `min(m,n)` collapses at least one direction and cannot be inverted on the whole
input space. This matrix rank is unrelated to tensor `ndim`.


In [ ]:
full = np.array([[1.0, 0.0], [0.0, 1.0]])
deficient = np.array([[1.0, 2.0], [2.0, 4.0]])
print("full rank:", np.linalg.matrix_rank(full))
print("deficient rank:", np.linalg.matrix_rank(deficient))

x1 = np.array([2.0, 0.0])
null_direction = np.array([-2.0, 1.0])
np.testing.assert_allclose(deficient @ x1, deficient @ (x1 + null_direction))


## 4. Basis changes and conditioning

A basis is a coordinate system. If columns of invertible $P$ are new basis vectors,
new coordinates are $P^{-1}x$. Nearly dependent basis vectors make $P$ ill-conditioned:
small numeric/input changes cause large coordinate changes.


In [ ]:
good_basis = np.array([[1.0, 0.0], [0.0, 1.0]])
bad_basis = np.array([[1.0, 1.0], [1.0, 1.000001]])
vector = np.array([2.0, 1.0])

for name, basis in {"good": good_basis, "bad": bad_basis}.items():
    coordinates = np.linalg.solve(basis, vector)
    reconstruction = basis @ coordinates
    print(name, "condition=", np.linalg.cond(basis), "coordinates=", coordinates)
    np.testing.assert_allclose(reconstruction, vector, atol=1e-9)


## Failure modes to recognize

- **Inner-dimension mismatch:** the proposed maps cannot be composed.
- **Weight transpose error:** an affine layer projects the wrong axes or fails its shape check.
- **Rank collapse:** distinct input directions become indistinguishable.
- **Ill-conditioning:** tiny perturbations create large solution or gradient changes.

These are diagnostic patterns, not trivia. When a future model fails, reduce the problem to the smallest example in this notebook and test the relevant invariant.


## Deliberate practice

1. Derive and implement a batched affine layer for `[B,T,Cin]` input.
2. Construct a 3×3 rank-2 matrix and find a nonzero vector in its null space.
3. Show numerically that matrix multiplication is associative but generally not commutative.

Do not merely rerun the provided cells. Make a prediction, change one variable, and write down why the result did or did not match your prediction.

## Exit ticket

**You are ready to continue when:** you can implement an affine map, predict every dimension, and explain rank loss and conditioning in geometric terms.

**Next:** 13 — Eigenvalues, SVD, and low-rank structure.
